# Fuzzy Booster: optimise SEO metadata with fuzzy matching and OpenAI

Companion notebook for the MLforSEO tutorial
[How to Automatically Optimize your SEO Metadata with FuzzyWuzzy and OpenAI in Google Colab](https://www.mlforseo.com/machine-learning-implementation-guides/onpage-seo/how-to-automatically-optimize-your-seo-metadata-with-fuzzywuzzy-and-openai-in-google-colab/).

What it does:
1. Pulls non-branded page + query data from Google Search Console and keeps the most-clicked query for each page.
2. Scrapes each page's `<title>`, meta description and first `<h1>`.
3. Cleans the text and scores how well each element matches the top query (`token_set_ratio`, 0-100).
4. Optionally asks an OpenAI model for a new title where the match is weak, then saves `fuzzy_booster.csv`.

**2026 refresh:** FuzzyWuzzy is now TheFuzz; this notebook uses [RapidFuzz](https://github.com/rapidfuzz/RapidFuzz), the maintained, faster library with the same scorers.
The Search Console sign-in no longer uses the retired "out-of-band" copy-paste flow, and the OpenAI key is read from a Colab secret instead of being pasted into the code.

**No Search Console access yet?** Run the setup cell and Step 1, then run the *Sample data* cell instead of Steps 2 and 3.

## Setup: install and import the libraries
`rapidfuzz` is the only library Colab doesn't already have. `openai` is upgraded to the current SDK.

In [ ]:
%pip install -q --upgrade rapidfuzz openai

In [ ]:
import os
import re
import datetime
import unicodedata
from collections import defaultdict
from urllib.parse import urlparse, parse_qs

import pandas as pd
import requests
import chardet
from bs4 import BeautifulSoup
from dateutil import relativedelta

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

from rapidfuzz import fuzz  # maintained replacement for FuzzyWuzzy: same token_set_ratio scorer
from openai import OpenAI

from google.oauth2 import service_account
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build

try:
    from google.colab import files  # upload/download helpers (Colab only)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## Step 1: Define query parameters
Pick the date range (Search Console keeps 16 months), your property, a regex for the home page, and a regex of branded terms to exclude.
Use `https://www.example.com/` for a URL-prefix property or `sc-domain:example.com` for a Domain property.

In [ ]:
# Select the date range
end_date = datetime.date.today()
start_date = end_date - relativedelta.relativedelta(months=16)

# Select the site
site = 'https://www.analistaseo.es/'  # https://www.example.com/ for a URL-prefix property or sc-domain:example.com for a Domain property

# Regex to match the home page
if site.startswith('sc-domain:'):
    home = r'^https?://(www\.)?' + re.escape(site.split(':', 1)[1]) + '/?$'
else:
    home = '^' + re.escape(site) + '$'

branded_queries = 'natzir|analistaseo|analista seo'  # Branded queries to exclude

# Request payload for the Google Search Console API
request = {
    'startDate': start_date.strftime('%Y-%m-%d'),
    'endDate': end_date.strftime('%Y-%m-%d'),
    'dimensions': ['page', 'query'],
    'dimensionFilterGroups': [
        {
            'filters': [
                {  # Excluding home page
                    'dimension': 'page',
                    'operator': 'excludingRegex',
                    'expression': home,
                },
                {  # Excluding branded queries
                    'dimension': 'query',
                    'operator': 'excludingRegex',
                    'expression': branded_queries,
                },
            ]
        }
    ],
    'rowLimit': 25000,  # Up to 25,000 rows per request
}

## Step 2: Upload your credentials file and get the Search Console data
Upload the JSON file you downloaded from Google Cloud (OAuth client ID of type **Desktop app**; a service-account key that has been added as a user in Search Console also works).

With an OAuth client, the cell prints a sign-in link. Open it, approve read-only access, and you'll land on a `http://localhost/...` page that **fails to load. That's expected.** Copy the full address from the browser bar and paste it into the box.
The cell then queries Search Console and keeps the most-clicked query for each page.

In [ ]:
# Upload the credentials file
if IN_COLAB:
    uploaded = files.upload()
    with open('credentials.json', 'wb') as f:
        f.write(uploaded[list(uploaded.keys())[0]])

SCOPES = ['https://www.googleapis.com/auth/webmasters.readonly']

def get_credentials(path='credentials.json', token_path='token.json'):
    # A service-account key: no browser sign-in needed
    with open(path) as f:
        if '"service_account"' in f.read():
            return service_account.Credentials.from_service_account_file(path, scopes=SCOPES)

    # An OAuth "Desktop app" client: reuse a saved token, otherwise sign in once
    creds = None
    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    if not creds or not creds.valid:
        flow = Flow.from_client_secrets_file(path, scopes=SCOPES, redirect_uri='http://localhost')
        auth_url, _ = flow.authorization_url(prompt='consent', access_type='offline')
        print(f'Please verify your account at the following URL:\n{auth_url}')
        reply = input('Paste the full localhost URL you were sent to (or just the code): ').strip()
        code = parse_qs(urlparse(reply).query).get('code', [reply])[0]
        flow.fetch_token(code=code)
        creds = flow.credentials
        with open(token_path, 'w') as token:
            token.write(creds.to_json())
    return creds

# Build the Search Console service
service = build('searchconsole', 'v1', credentials=get_credentials())

# Execute the request to the Search Console API
print('Getting Google Search Console data...')
response = service.searchanalytics().query(siteUrl=site, body=request).execute()

# Parse the JSON response
scDict = defaultdict(list)

for row in response.get('rows', []):
    scDict['page'].append(row['keys'][0] or 0)
    scDict['query'].append(row['keys'][1] or 0)
    scDict['clicks'].append(row['clicks'] or 0)
    scDict['ctr'].append(row['ctr'] or 0)
    scDict['impressions'].append(row['impressions'] or 0)
    scDict['position'].append(row['position'] or 0)

# Create a DataFrame from the parsed data
df = pd.DataFrame(data=scDict)

# Convert data types and adjust values
df['clicks'] = df['clicks'].astype('int')
df['ctr'] = df['ctr'] * 100
df['impressions'] = df['impressions'].astype('int')
df['position'] = df['position'].round(2)
df.sort_values('clicks', inplace=True, ascending=False)

# Keep the most clicked query for each page
df = df.drop_duplicates(subset=['page'], keep='first')
df

### Sample data (optional): run this instead of Steps 2 and 3
Loads a small made-up Spanish dataset that already has the Search Console columns plus the scraped `title`, `meta` and `h1`, so you can try Steps 4-6 without credentials.

In [ ]:
SAMPLE_CSV = 'https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/seo-metadata-fuzzy-matching-openai/sample_gsc_metadata.csv'
df = pd.read_csv('sample_gsc_metadata.csv' if os.path.exists('sample_gsc_metadata.csv') else SAMPLE_CSV)
df = df.sort_values('clicks', ascending=False).drop_duplicates(subset=['page'], keep='first')
df

## Step 3: Scrape metadata with BeautifulSoup
Fetches each page and pulls the `<title>`, meta description and first `<h1>`. Check the site's terms before scraping pages you don't own.

In [ ]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; FuzzyBooster/2.0; +https://www.mlforseo.com/)'}

# Function to extract metadata from a URL
def get_meta(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=20)
        encoding = chardet.detect(response.content)['encoding']
        if encoding:
            page_content = response.content.decode(encoding, errors='replace')
        else:
            page_content = response.content
        soup = BeautifulSoup(page_content, 'html.parser')
        title = soup.find('title').get_text(strip=True) if soup.find('title') else 'No title'  # Get the title
        description = soup.select_one('meta[name="description"]')
        meta = description.get('content', '').strip() if description else 'No meta description'  # Get the meta description
        h1 = soup.find('h1').get_text(strip=True) if soup.find('h1') else 'No h1'  # Get the first h1
        return title, meta, h1
    except Exception as e:
        print(f'Could not fetch {url}: {e}')
        return 'Error', 'Error', 'Error'

# Apply the function and add the results to the DataFrame
df['title'], df['meta'], df['h1'] = zip(*df['page'].apply(get_meta))
df

## Step 4: Clean and prepare text data
Set `lang` to your site's language (any language NLTK has stopwords for, e.g. `'english'`, `'spanish'`, `'german'`).
The cleaner lowercases, removes numbers, punctuation and accents, normalises whitespace and drops stopwords.

In [ ]:
lang = 'spanish'  # Language for stopwords: change this to the language of your website

In [ ]:
def strip_accents(text):
    return unicodedata.normalize('NFKD', text).encode('ascii', errors='ignore').decode('utf-8')

# Stopwords with and without accents, because clean_text removes accents before filtering
stop_words = set(stopwords.words(lang))
stop_words |= {strip_accents(word) for word in stop_words}

# Function to clean text
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = strip_accents(text)

    words = text.split()
    words_cleaned = [word for word in words if word not in stop_words]
    return ' '.join(words_cleaned)

# Clean specified columns
columns_to_clean = ['title', 'meta', 'h1', 'query']

for column in columns_to_clean:
    df[column + '_clean'] = df[column].apply(clean_text)

df

## Step 5: Compute similarity using fuzzy matching
`token_set_ratio` compares the words two strings share, ignoring order and repeats. 100 means every word of the shorter string is in the longer one.
You get `title_similarity`, `meta_similarity` and `h1_similarity` for each page.

In [ ]:
columns = ['title_clean', 'meta_clean', 'h1_clean']

for col in columns:
    similarity = []
    for index, row in df.iterrows():
        sim = fuzz.token_set_ratio(row['query_clean'], row[col])
        similarity.append(round(sim))
    df[f'{col}_similarity'] = similarity

# Rename columns for clarity
df.rename(columns=lambda x: x.replace('_clean_similarity', '_similarity') if x.endswith('_clean_similarity') else x, inplace=True)
columns_to_drop = [col for col in df.columns if '_clean' in col]
df.drop(columns=columns_to_drop, inplace=True)

df

## Step 6: Generate new titles using OpenAI (optional)
Pages whose title scores 60 or below against their top query get a new title suggestion. Adjust the threshold to taste.

Add your OpenAI key as a Colab secret named `OPENAI_API_KEY` (key icon in the left sidebar, then allow this notebook access). Locally, set the `OPENAI_API_KEY` environment variable.
API calls cost money, so start with a small dataset.

In [ ]:
filtered_df = df[df['title_similarity'] <= 60]
filtered_df

In [ ]:
# Configure the OpenAI API: the key comes from a Colab secret or an environment variable, never from the code
if IN_COLAB and not os.environ.get('OPENAI_API_KEY'):
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

client = OpenAI()  # reads OPENAI_API_KEY

MODEL = 'gpt-4o-mini'  # small, cheap current model; swap for a larger one if you prefer
TEMPERATURES = [0.0]

In [ ]:
PROMPT_TEMPLATE = """Task: Generate a new title that meets the following criteria:
1. The title cannot be longer than 60 characters.
2. The title must include the specified query, represented in the template as '{query}'.
3. The title must be written in the specified language, represented in the template as '{lang}'.
4. No words should be repeated in the title.
5. Use the entities found in the given 'current title' '{title}' to construct the new title.
Please ensure that your title meets all of the criteria. Reply with the title only.
"""

In [ ]:
# Function to generate a new title using OpenAI
def generar_titulo(prompt, temperature):
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return completion.choices[0].message.content.strip()

# Generate new titles for each filtered row
df['new_title'] = ''
for index, row in filtered_df.iterrows():
    query = row['query']
    current_title = row['title']
    prompt = PROMPT_TEMPLATE.format(query=query, lang=lang, title=current_title)
    new_title = generar_titulo(prompt, TEMPERATURES[0])
    df.at[index, 'new_title'] = new_title

# Function to strip quotes from the beginning and end of a string
def strip_quotes(title):
    title = str(title)
    if len(title) > 1 and title[0] in '"\'' and title[-1] == title[0]:
        return title[1:-1]
    return title

# Apply the function to the 'new_title' column
df['new_title'] = df['new_title'].apply(strip_quotes)

# Save the results to a CSV file
df.to_csv('fuzzy_booster.csv', index=False)

# Download the CSV file
if IN_COLAB:
    files.download('fuzzy_booster.csv')

df

## Next
Apply the same process to `h1` and `meta`: filter on `h1_similarity` or `meta_similarity` and adjust the prompt (meta descriptions are usually kept under about 155 characters).